# Instruction tuning: teaching the base model to answer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/tinystories-ARM-model/blob/main/sft_colab.ipynb)

The base model (124M, FineWeb-Edu + Cosmopedia, validation loss 2.903) has only read documents: ask it
*"What is the water cycle?"* and it writes more questions, or the start of an article. **Supervised fine-tuning
(SFT)** shows it conversations instead, so it learns the shape of an answer.

- **Data**: `HuggingFaceTB/smol-smoltalk`, 460K conversations written for small models.
- **Format** (plain text, no new tokens): `User: …` / `Assistant: …`, every answer ending with `<|endoftext|>`
  so generation knows where to stop.
- **Loss**: only on the assistant's tokens. The model learns to answer, not to write questions.
- **Packing**: whole conversations in 512-token windows (never split).

The code is `sft.py`; this notebook runs it on a GPU and keeps the result in Google Drive.

## Step 0 · Get the code

In [ ]:
# @title Clone the repository and install the dependencies
import os, sys

REPO_URL = "https://github.com/a-ramakrishnan/tinystories-ARM-model.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    CLONE_DIR = "/content/tinystories-ARM-model"
    if not os.path.exists(CLONE_DIR):
        !git clone -q {REPO_URL} {CLONE_DIR}
    else:
        !git -C {CLONE_DIR} pull -q
    %cd {CLONE_DIR}
    !pip install -q -r requirements.txt
assert os.path.exists("sft.py"), "Open this notebook from the tinystories-ARM-model folder."
PROJECT_DIR = os.getcwd()
print("Project folder:", PROJECT_DIR)

## Step 1 · Settings

| Mode | Data | Training | Time on an L4 |
|---|---|---|---|
| `smoke` | 5,000 conversations | 100 updates | a few minutes: checks everything works |
| `full` | all 460K conversations | one pass (about 13,000 updates of 32 × 512) | about 1 h 15 min |

**Base weights**: the base model's training run in Drive (`MyDrive/tinystories-runs/mix-…`). Its config is in
the repository (`sft_configs/base-14ed26.json`), in case the run folder doesn't have one.

In [ ]:
# @title Settings
run_mode = "full"  # @param ["smoke", "full"]
learning_rate = 2e-4  # @param {type:"number"}
batch_size = 32  # @param {type:"integer"}
keep_in_drive = True  # @param {type:"boolean"}
base_run = "/content/drive/MyDrive/tinystories-runs/mix-baominhtran__mini-+huggingfacetb__cos+huggingfacetb__cos+huggingfacetb__cos+huggingfacetb__cos+huggingfacetb__cos-e3deeec9"  # @param {type:"string"}

import json, shutil, time, torch
from train_config import setup_device

dev = setup_device()
print(f"Device: {dev.device} ({dev.dtype})" + (f" · {torch.cuda.get_device_name(0)}" if dev.device == "cuda" else ""))
LIMIT = 5000 if run_mode == "smoke" else None
STEPS = 100 if run_mode == "smoke" else None
if IN_COLAB and keep_in_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/sft-runs"
else:
    OUT_ROOT = os.path.join(PROJECT_DIR, "outputs", "sft")
RUN_DIR = os.path.join(OUT_ROOT, f"smoltalk-{run_mode}-{time.strftime('%Y%m%d-%H%M')}")
os.makedirs(RUN_DIR, exist_ok=True)

# the base model: weights from Drive, config from the run folder or the repository
BASE_DIR = "/content/base-model"
os.makedirs(BASE_DIR, exist_ok=True)
src = os.path.join(base_run, "best_model_params.pt")
assert os.path.exists(src), f"Base weights not found: {src}"
shutil.copy(src, os.path.join(BASE_DIR, "best_model_params.pt"))
cfg_src = os.path.join(base_run, "model_config.json")
shutil.copy(cfg_src if os.path.exists(cfg_src) else "sft_configs/base-14ed26.json", os.path.join(BASE_DIR, "model_config.json"))
print("Base model:", json.load(open(os.path.join(BASE_DIR, "model_config.json"))).get("best_val"), "validation loss")
print("Run folder:", RUN_DIR)

## Step 2 · Tokenize the conversations

Each conversation becomes token ids plus a mask that is 1 on the assistant's answers. Conversations longer than
512 tokens are cut after the last answer that fits; those whose first answer doesn't fit are dropped.

In [ ]:
# @title Prepare the data
import numpy as np
from sft import prepare, DATA_DIR

data_dir = DATA_DIR + ("-smoke" if LIMIT else "")
cache = os.path.join(OUT_ROOT, "data", os.path.basename(data_dir))
if os.path.exists(os.path.join(cache, "info.json")):
    shutil.copytree(cache, data_dir, dirs_exist_ok=True)
    print("Copied the tokenized conversations from Drive:", json.load(open(os.path.join(data_dir, "info.json")))["train"])
else:
    prepare(LIMIT, out_dir=data_dir)
    if OUT_ROOT.startswith("/content/drive"):
        shutil.copytree(data_dir, cache, dirs_exist_ok=True)

In [ ]:
# @title Look at one training window (the learned parts in [brackets])
from tokenization import get_tokenizer
enc = get_tokenizer()
ids, mask = np.load(os.path.join(data_dir, "train_ids.npy"), mmap_mode="r")[0].tolist(), np.load(os.path.join(data_dir, "train_mask.npy"), mmap_mode="r")[0].tolist()
out = []
for i, (t, m) in enumerate(zip(ids, mask)):
    if m and (i == 0 or not mask[i - 1]): out.append("[")
    out.append(enc.decode([t]))
    if m and (i == len(ids) - 1 or not mask[i + 1]): out.append("]")
print("".join(out)[:1500])

## Step 3 · Fine-tune

It reports the **held-out answer loss** (cross-entropy on the assistant tokens of the test conversations) before
training and every 500 updates, keeping the best checkpoint. Learning rate 2e-4 with a short warm-up, cosine to a
tenth: about a sixth of the pre-training peak, so the model learns the format without forgetting what it read.

In [ ]:
# @title Train
from sft import train
summary = train(BASE_DIR, RUN_DIR, steps=STEPS, batch_size=batch_size, lr=learning_rate, min_lr=learning_rate / 10,
                eval_every=50 if STEPS else 500, data_dir=data_dir, base_id="mini-fineweb-edu-202611130718-14ed26")

In [ ]:
# @title Loss curves
import matplotlib.pyplot as plt
h = summary["sft"]["history"]
plt.figure(figsize=(8, 4))
plt.plot([x[0] for x in h["train"]], [x[1] for x in h["train"]], alpha=0.5, label="training batches")
plt.plot([x[0] for x in h["test"]], [x[1] for x in h["test"]], "o-", label="held-out answers")
plt.xlabel("update"); plt.ylabel("loss (answer tokens)"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(RUN_DIR, "loss.png"), dpi=120, bbox_inches="tight"); plt.show()

## Step 4 · Ask it something

In [ ]:
# @title Chat
question = "How do plants make their food?"  # @param {type:"string"}
temperature = 0.0  # @param {type:"number"}
from inference import load_model, load_model_config
from sft import answer
config, _ = load_model_config(os.path.join(RUN_DIR, "model_config.json"))
chat_model = load_model(config, os.path.join(RUN_DIR, "best_model_params.pt"), dev.device)
print("User:", question)
print("Assistant:", answer(chat_model, enc, question, dev.device, temperature=temperature, top_k=50 if temperature > 0 else None))

## Step 5 · Package the results

The weights, config (with `"chat": true`), loss chart and samples in one zip, in the run folder. Tick `download`
to also download it to your computer; put it in the project folder on the Mac and add it to the model library.

In [ ]:
# @title Make the zip
download = False  # @param {type:"boolean"}
import zipfile
zip_path = os.path.join(RUN_DIR, f"sft-{os.path.basename(RUN_DIR)}.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for f in ("best_model_params.pt", "model_config.json", "run.json", "loss.png"):
        if os.path.exists(os.path.join(RUN_DIR, f)):
            z.write(os.path.join(RUN_DIR, f), f)
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
if IN_COLAB and download:
    from google.colab import files
    files.download(zip_path)